# Overview

1. **Imports and Setup**
2. **Load the Data**
3. **Drop Identifier and High-Missing Columns**
4. **Drop Rows Missing the Target Variable**
5. **Fill Remaining Missing Values**
6. **Remove Outliers**
7. **Encode Categorical Variables**
8. **Explore Target Distribution and Correlations**
9. **Demonstrate Feature Scaling**
10. **Save Preprocessed Data and Final Preview**

---

**Edit the configuration variables in Step 1** to match your dataset, then run each cell in order.

The output file is formatted for AWS SageMaker Linear Learner: no header row, target variable in the first column.

# STEP 1: IMPORTS AND SETUP

In [ ]:
# Install any packages not included in the SageMaker environment
!pip install seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew
from sklearn.preprocessing import (
    PowerTransformer, 
    StandardScaler, 
    MinMaxScaler, 
    PolynomialFeatures
)

# ============================================================
# CHANGE THESE VALUES FOR YOUR DATASET
# ============================================================

file_path = "AmesHousing.csv"          # Your CSV file name
target_col = "SalePrice"               # Column you want to predict

# ============================================================
# DO NOT CHANGE ANYTHING BELOW THIS LINE
# ============================================================

# Auto-generate output filename from input filename
output_file = file_path.replace(".csv", "_preprocessed.csv")

print("Configuration set:")
print(f"  Input file:      {file_path}")
print(f"  Target column:   {target_col}")
print(f"  Output file:     {output_file}")

## Why These Imports?

- **pandas**: For data handling  
- **numpy**: For numeric calculations  
- **matplotlib/seaborn**: For plotting and data visualization  
- **sklearn.preprocessing**: For transformations (scaling and power transforms)  


# STEP 2: LOAD THE DATA

In [ ]:
df = pd.read_csv(file_path)

# Show the first 5 rows (to see what the data looks like)
print("Data Loaded Successfully!\n")
print(df.head())

# Shape tells us how many rows and columns
print("\nShape of the dataset:", df.shape)

# info() shows column names, data types, and where missing values might be
df.info()

## Note

- `df.head()`: Lets us see the top rows to get a feel for what columns we have.  
- `df.info()`: Shows how many non-missing (non-null) entries each column has.  


## From the Output:

### Missing Values:
- Look for columns with fewer non-null entries than the total row count. These columns have missing data.
- Columns with very few non-null entries (e.g., less than 10% of rows) will be dropped in Step 3.

### Data Types:
- **int64**: Usually represents whole numbers (e.g., counts, years).  
- **float64**: Often represents numeric data that can have decimals (e.g., measurements, prices).  
- **object**: Typically represents text or categorical data (e.g., names, categories).  

# STEP 3: DROP IDENTIFIER AND HIGH-MISSING COLUMNS

First, we **auto-detect and drop identifier columns**. A column is flagged as an identifier if every value is unique (cardinality equals the row count) and it is not the target variable. These are row IDs with no predictive value that can mislead the algorithm.

Then we drop any column where more than **30%** of its values are missing.
You can choose a different threshold based on your needs.

In [ ]:
def drop_identifiers_and_high_missing(dataframe, target_column, threshold=0.3):
    """
    1. Auto-detect and drop identifier columns (all unique values, not the target).
    2. Drop any column where the fraction of missing values exceeds the threshold.
    """
    # Auto-detect identifier columns: every value is unique and it's not the target
    id_cols = [
        c for c in dataframe.columns
        if c != target_column and dataframe[c].nunique() == len(dataframe)
    ]

    if id_cols:
        dataframe = dataframe.drop(columns=id_cols)
        print(f"Auto-detected and dropped identifier columns: {id_cols}")
    else:
        print("No identifier columns detected.")

    # Drop columns with too many missing values
    missing_fraction = dataframe.isnull().mean()
    cols_to_drop = missing_fraction[missing_fraction > threshold].index
    print(f"Columns to drop (>{threshold*100}% missing): {list(cols_to_drop)}")
    dataframe = dataframe.drop(columns=cols_to_drop)

    print(f"\nRemaining shape: {dataframe.shape}")
    return dataframe

df = drop_identifiers_and_high_missing(df, target_column=target_col, threshold=0.3)

# STEP 4: DROP ROWS MISSING THE TARGET VARIABLE

We cannot train a model if our target variable is missing.

In [ ]:
def drop_missing_target(dataframe, target_column):
    """
    Drop rows that have missing values for the target column.
    """
    if target_column not in dataframe.columns:
        print(f"Target column '{target_column}' not found. Skipping.")
        return dataframe

    before = len(dataframe)
    dataframe = dataframe.dropna(subset=[target_column])
    after = len(dataframe)
    print(f"Dropped {before - after} rows that had missing {target_column}.")

    return dataframe

df = drop_missing_target(df, target_col)

# STEP 5: FILL REMAINING MISSING VALUES

Now we decide how to fill in the other missing values. A simple approach:

- **Numeric columns**: Fill with the median value.
- **Categorical (string/object) columns**: Fill with `"Missing"`.

In [ ]:
def fill_missing_values(dataframe):
    """
    Fills missing numeric values with the median,
    and missing categorical values with 'Missing'.
    """
    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns
    categorical_cols = dataframe.select_dtypes(exclude=[np.number]).columns

    # Numeric -> median
    for col in numeric_cols:
        if dataframe[col].isnull().any():
            dataframe[col] = dataframe[col].fillna(dataframe[col].median())

    # Categorical -> "Missing"
    for col in categorical_cols:
        if dataframe[col].isnull().any():
            dataframe[col] = dataframe[col].fillna("Missing")

    return dataframe

### Check Missing Values Before:
- Let us see how many missing values exist in total.  
- We will also inspect the columns with the most missing values.

In [ ]:
# Count total missing values across the entire DataFrame
missing_before = df.isnull().sum().sum()
print(f"Total missing values (BEFORE): {missing_before}")

# Show columns with the most missing values
missing_by_col = df.isnull().sum()
missing_cols = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
if len(missing_cols) > 0:
    print(f"\nColumns with missing values ({len(missing_cols)} columns):")
    for col, count in missing_cols.head(10).items():
        print(f"  {col}: {count} missing ({count/len(df)*100:.1f}%)")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more columns")
else:
    print("\nNo missing values found.")

### Fill Missing Values

In [ ]:
df = fill_missing_values(df)

### Check Missing Values After

In [ ]:
# Count total missing values across the entire DataFrame again
missing_after = df.isnull().sum().sum()
print(f"\nTotal missing values (AFTER): {missing_after}")

# Verify all missing values are filled
if missing_after == 0:
    print("All missing values have been filled successfully.")
else:
    missing_remaining = df.isnull().sum()
    still_missing = missing_remaining[missing_remaining > 0]
    print(f"\nWarning: {len(still_missing)} column(s) still have missing values:")
    for col, count in still_missing.items():
        print(f"  {col}: {count} missing")

# STEP 6: REMOVE OUTLIERS

This step **auto-detects** the numeric feature with the most extreme outliers using the IQR (Interquartile Range) method, then removes rows that exceed the upper fence (`Q3 + 1.5 * IQR`).

Extreme outliers are rare and can skew linear models.

In [ ]:
def remove_outliers(dataframe, target_column):
    """
    Auto-detect the numeric column (excluding target) with the most extreme
    outliers using IQR, then remove rows exceeding the upper fence.
    Shows a before/after boxplot.
    """
    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c != target_column]

    if len(feature_cols) == 0:
        print("No numeric feature columns found. Skipping outlier removal.")
        return dataframe

    # Find the column with the most outliers above the upper fence (Q3 + 1.5*IQR)
    worst_col = None
    worst_count = 0
    worst_upper = 0

    for col in feature_cols:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_fence = Q3 + 1.5 * IQR
        n_outliers = (dataframe[col] > upper_fence).sum()
        if n_outliers > worst_count:
            worst_col = col
            worst_count = n_outliers
            worst_upper = upper_fence

    if worst_count == 0:
        print("No outliers detected in any numeric column. Skipping.")
        return dataframe

    print(f"Auto-detected outlier column: '{worst_col}' ({worst_count} outliers above {worst_upper:.1f})")

    df_before = dataframe.copy()

    # Boxplot before
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    sns.boxplot(x=df_before[worst_col])
    plt.title(f"{worst_col} - Before")

    # Remove the outliers
    before_count = len(dataframe)
    dataframe = dataframe[dataframe[worst_col] <= worst_upper]
    after_count = len(dataframe)

    # Boxplot after
    plt.subplot(1, 2, 2)
    sns.boxplot(x=dataframe[worst_col])
    plt.title(f"{worst_col} - After")
    plt.tight_layout()
    plt.show()

    print(f"Removed {before_count - after_count} rows where {worst_col} > {worst_upper:.1f}")

    return dataframe

df = remove_outliers(df, target_column=target_col)

# Boxplots: Before and After Removing Outliers

## 1. Boxplot Before Removing Outliers (Left Plot)
The left boxplot shows the distribution of the outlier column prior to outlier removal.

- Data points beyond the whiskers are identified as potential outliers.
- These extreme values could distort analyses such as linear regression by introducing skewness or bias.

## 2. Boxplot After Removing Outliers (Right Plot)
The right boxplot shows the same data after filtering out values exceeding the configured limit.

## 3. Key Takeaways:
- Removing extreme outliers minimizes the risk of skewness or bias, especially when the outliers are rare or atypical.
- This adjustment improves the reliability of statistical methods, like regression, by ensuring the analysis reflects the core patterns of the dataset without undue influence from non-representative data points.
- By refining the dataset in this way, models are more likely to produce accurate and meaningful insights.

## Note

In reality, outlier handling depends on context.  
Here, we are removing extreme values simply as an example.

# STEP 7: ENCODE CATEGORICAL VARIABLES

Machine learning models like linear regression typically prefer numeric data.
So we must convert ("encode") any categorical columns.

- **Frequency Encoding**: For columns with many categories (over 10), we replace each category with how often it appears in the dataset (a fraction from 0 to 1).
- **One-Hot Encoding**: For columns with few categories, we create new columns of 0s and 1s for each category (minus one category to avoid redundancies).

In [ ]:
def encode_categorical_features(dataframe, freq_threshold=10):
    """
    If a column has more than `freq_threshold` unique categories,
    we use frequency encoding (replacing each category with its fraction of occurrences).
    Otherwise, we do one-hot encoding (creating new 0/1 columns for each category).

    This version prints out:
      1) Which columns were frequency-encoded vs. one-hot-encoded
      2) A small sample (first 5 rows) of one frequency-encoded column
         and one one-hot-encoded column (if any exist).
    """
    # Identify categorical columns (object dtype)
    cat_cols = dataframe.select_dtypes(include=["object"]).columns
    
    one_hot_frames = []
    freq_frames = {}
    
    # Keep track of columns that were encoded by which method
    freq_encoded_columns = []
    one_hot_encoded_columns = []
    
    # Decide how to encode each categorical column
    for col in cat_cols:
        unique_count = dataframe[col].nunique()
        
        if unique_count > freq_threshold:
            # Frequency encoding
            freq_map = dataframe[col].value_counts(normalize=True)
            freq_frames[col + "_freq"] = dataframe[col].map(freq_map)
            freq_encoded_columns.append(col)
        else:
            # One-hot encoding
            dummies = pd.get_dummies(dataframe[col], prefix=col, drop_first=True, dtype=int)
            one_hot_frames.append(dummies)
            one_hot_encoded_columns.append(col)

    # Merge frequency-encoded columns back
    if freq_frames:
        freq_df = pd.DataFrame(freq_frames, index=dataframe.index)
        dataframe = pd.concat([dataframe, freq_df], axis=1)

    # Merge one-hot-encoded columns back
    if one_hot_frames:
        ohe_df = pd.concat(one_hot_frames, axis=1)
        dataframe = pd.concat([dataframe, ohe_df], axis=1)

    # Drop original categorical columns (replaced by numeric ones)
    dataframe.drop(columns=cat_cols, inplace=True)
    
    # ---------------------------
    # PRINT A SUMMARY OF RESULTS
    # ---------------------------
    print("\nEncoding Summary:")

    # Frequency-encoded columns
    if freq_encoded_columns:
        print(f"  Frequency-encoded columns (>{freq_threshold} unique categories):")
        for col in freq_encoded_columns:
            print(f"    - {col}")
    else:
        print(f"  No columns had more than {freq_threshold} unique categories.")

    # One-hot-encoded columns
    if one_hot_encoded_columns:
        print(f"\n  One-hot-encoded columns (<= {freq_threshold} unique categories):")
        for col in one_hot_encoded_columns:
            print(f"    - {col}")
    else:
        print(f"  No columns had {freq_threshold} or fewer unique categories.")

    # ---------------------------
    # SHOW EXAMPLES OF TRANSFORMED COLUMNS
    # ---------------------------
    # 1) Example of a frequency-encoded column
    if freq_encoded_columns:
        freq_example = freq_encoded_columns[0]               # just pick the first column we freq-encoded
        freq_example_col = freq_example + "_freq"            # this is how we named it above
        if freq_example_col in dataframe.columns:
            print(f"\nExample of frequency-encoded column: '{freq_example_col}'")
            display(dataframe[[freq_example_col]].head(5))
    
    # 2) Example of a one-hot-encoded column
    if one_hot_encoded_columns:
        oh_example = one_hot_encoded_columns[0]              # pick the first column we one-hot-encoded
        # Our new one-hot columns for this feature start with oh_example + "_"
        oh_cols = [c for c in dataframe.columns if c.startswith(oh_example + "_")]
        if oh_cols:
            print(f"\nExample of one-hot-encoded columns from '{oh_example}': {oh_cols}")
            display(dataframe[oh_cols].head(5))

    return dataframe


# Usage Example
df = encode_categorical_features(df, freq_threshold=10)


# STEP 8: EXPLORE TARGET DISTRIBUTION AND CORRELATIONS

Before saving, we explore two important characteristics of our preprocessed data:

1. **Target Distribution**: Numeric targets are often right-skewed. We demonstrate how a PowerTransformer can normalize the distribution (for educational purposes only, the saved file keeps the original target values).
2. **Correlation Analysis**: We check how strongly each feature relates to the target variable.

In [ ]:
def explore_target_distribution(dataframe, target_column):
    """
    Visualizes the target column distribution and demonstrates how a
    PowerTransformer reduces skew. This is for educational purposes only
    and does not modify the DataFrame.
    """
    if target_column not in dataframe.columns:
        print(f"Target column '{target_column}' not found. Skipping.")
        return

    target_values = dataframe[target_column].copy()
    skew_before = skew(target_values)
    print(f"Current {target_column} skew: {skew_before:.2f}")

    # Apply Yeo-Johnson transform to a COPY (for demonstration only)
    transformer = PowerTransformer(method="yeo-johnson")
    target_transformed = transformer.fit_transform(
        target_values.values.reshape(-1, 1)
    ).flatten()
    skew_after = skew(target_transformed)
    print(f"After PowerTransformer skew: {skew_after:.2f}")

    print(f"\nNote: This transformation is shown for educational purposes.")
    print(f"The saved preprocessed file keeps the original {target_column} values.")

    # Compare the distributions
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(target_values, kde=True)
    plt.title(f"{target_column} - Original Distribution")

    plt.subplot(1, 2, 2)
    sns.histplot(target_transformed, kde=True)
    plt.title(f"{target_column} - After PowerTransformer")

    plt.tight_layout()
    plt.show()

explore_target_distribution(df, target_col)

## Interpretation of Target Distribution

If the original target distribution is **right-skewed**, most values are clustered at the lower end with a long tail of higher values. A PowerTransformer (Yeo-Johnson method) can normalize this to near-zero skew, which improves regression model performance.

**Note:** The saved preprocessed file keeps the **original target values**. Downstream notebooks can apply their own transformations as needed.

---

## Correlation Analysis

Correlation helps us see how strongly each feature is related to our target. Features with higher absolute correlation are more predictive.

In [ ]:
def correlation_analysis(dataframe, target_column="SalePrice"):
    """
    Prints out the correlation of each column with the target,
    and then shows a full correlation heatmap.
    """
    if target_column not in dataframe.columns:
        print(f"Target column '{target_column}' not found. Skipping correlation analysis.")
        return

    corr_matrix = dataframe.corr(numeric_only=True)
    target_corr = corr_matrix[target_column].sort_values(ascending=False)

    print("\nTop 10 features MOST positively correlated with target:")
    print(target_corr[1:11])  # skip the target itself at index 0

    print("\nTop 10 features MOST negatively correlated with target:")
    print(target_corr[-10:])

    # Plot a heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap="coolwarm", annot=False, square=True)
    plt.title("Correlation Matrix Heatmap")
    plt.show()

correlation_analysis(df, target_column=target_col)

# STEP 9: DEMONSTRATE FEATURE SCALING

Scaling features using **StandardScaler** can sometimes help linear regression, especially if the ranges of features differ greatly.

This demonstration shows how one column looks before and after scaling. The saved preprocessed file keeps the original (unscaled) values because SageMaker's Linear Learner handles scaling internally.

In [ ]:
def scale_features_demo(dataframe, target_column, col_to_plot=None):
    """
    Shows how one column looks before and after StandardScaler.
    Works on a copy of the data (does not modify the original DataFrame).
    If col_to_plot is None, automatically picks the numeric column with the highest variance.
    """
    # Separate the features from the target
    if target_column in dataframe.columns:
        X = dataframe.drop(columns=[target_column])
    else:
        X = dataframe.copy()

    # Auto-select a column to plot if none specified
    if col_to_plot is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            col_to_plot = X[numeric_cols].var().idxmax()
            print(f"Auto-selected column with highest variance: '{col_to_plot}'")
        else:
            print("No numeric columns available. Cannot demonstrate scaling.")
            return

    # Fit the StandardScaler
    scaler_std = StandardScaler()
    X_std = scaler_std.fit_transform(X)

    # Convert to a DataFrame for easy plotting
    X_std_df = pd.DataFrame(X_std, columns=X.columns, index=X.index)

    # Plot original vs. standard-scaled distributions for col_to_plot
    if col_to_plot in X.columns:
        plt.figure(figsize=(12, 4))

        # Original
        plt.subplot(1, 2, 1)
        sns.histplot(X[col_to_plot], kde=True, color="skyblue")
        plt.title(f"{col_to_plot} - Original")

        # StandardScaled
        plt.subplot(1, 2, 2)
        sns.histplot(X_std_df[col_to_plot], kde=True, color="orange")
        plt.title(f"{col_to_plot} - StandardScaled")

        plt.tight_layout()
        plt.show()
    else:
        print(f"Column '{col_to_plot}' was not found. Cannot plot distribution.")

scale_features_demo(df, target_column=target_col)

# STEP 10: SAVE PREPROCESSED DATA AND FINAL PREVIEW

Now that we have cleaned, filled missing values, removed outliers, and encoded all categorical variables,
we save the preprocessed DataFrame to a CSV file.

**Copy your preprocessed CSV into each Linear Learner notebook directory before running those labs.**

In [ ]:
def save_preprocessed_data(dataframe, output_file, target_column):
    """
    Save the preprocessed DataFrame to a CSV file for use by downstream notebooks.
    SageMaker's built-in algorithms expect CSV with no header and the target
    variable in the first column, so we format the output accordingly.
    """
    # Move target column to the first position (SageMaker CSV format requirement)
    if target_column in dataframe.columns:
        cols = [target_column] + [c for c in dataframe.columns if c != target_column]
        dataframe = dataframe[cols]
        print(f"Moved '{target_column}' to the first column (SageMaker requirement).")
    else:
        print(f"WARNING: Target column '{target_column}' not found in DataFrame!")
        print("The output file will not have the target in the first column.")

    # Save without header or index (SageMaker CSV format: no headers)
    dataframe.to_csv(output_file, index=False, header=False)
    print(f"Preprocessed data saved to: {output_file}")
    print(f"Shape: {dataframe.shape[0]} rows x {dataframe.shape[1]} columns")
    print(f"Format: No header row, target ({target_column}) in first column")

save_preprocessed_data(df, output_file=output_file, target_column=target_col)

print("\nPreview of the processed DataFrame:")
display(df.head())

print("\nFinal shape of processed data:", df.shape)